# 04 · Machine Learning & AI Foundations

Quantum *Machine Learning* (notebook 03) borrows almost all of its vocabulary — *features, labels, training, loss, gradients, overfitting* — from **classical** machine learning. To understand Quantum AI you must first understand ordinary AI. This notebook is a compact but solid grounding in ML & AI, with runnable examples.

We cover:

1. AI vs. Machine Learning vs. Deep Learning — what the words mean
2. The three kinds of learning (supervised, unsupervised, reinforcement)
3. The core ML workflow (data → model → train → evaluate)
4. A real classifier you train yourself (scikit-learn)
5. How neural networks learn: weights, loss, gradient descent
6. A tiny neural network from scratch in NumPy
7. Overfitting, generalisation, and why data matters
8. How all of this connects to **Quantum** ML

In [ ]:
# Run once.
%pip install scikit-learn numpy matplotlib

## 1. AI vs. ML vs. Deep Learning

These terms are nested, like Russian dolls:

```
┌─────────────────────────────────────────────┐
│ Artificial Intelligence (AI)                 │  Any technique that makes
│  machines act 'intelligently'                │  machines mimic human-like reasoning
│  ┌────────────────────────────────────────┐ │
│  │ Machine Learning (ML)                   │ │  Systems that LEARN patterns
│  │  from data instead of being hand-coded  │ │  from examples, not rules
│  │  ┌───────────────────────────────────┐ │ │
│  │  │ Deep Learning (DL)                │ │ │  ML using many-layered
│  │  │  neural networks (GPT, vision…)   │ │ │  neural networks
│  │  └───────────────────────────────────┘ │ │
│  └────────────────────────────────────────┘ │
└─────────────────────────────────────────────┘
```

- **AI** is the broad goal: machines that perform tasks needing 'intelligence'.
- **ML** is the dominant *approach* to AI today: instead of writing explicit rules, we show the computer examples and let it *learn* the rule.
- **Deep Learning** is ML using **neural networks** with many layers — behind image recognition, speech, and large language models like the one you're talking to.

> **The key shift:** Traditional programming is `rules + data → answers`. Machine learning flips it: `data + answers → rules` (the trained model).

## 2. The three kinds of machine learning

| Type | You give it… | It learns to… | Example |
|------|--------------|---------------|---------|
| **Supervised** | inputs **+ correct labels** | predict labels for new inputs | Spam detection, image classification, the quantum classifier in nb 03 |
| **Unsupervised** | inputs **only** | find structure / groups | Customer segmentation, anomaly detection |
| **Reinforcement** | an environment + rewards | take actions that maximise reward | Game-playing AI, robotics, self-driving |

Most of this notebook (and notebook 03) is **supervised learning** — the most common starting point.

## 3. The core ML workflow

Almost every supervised ML project follows the same loop:

```
1. DATA        Collect & clean examples (features X, labels y)
2. SPLIT       Hold back a test set the model never sees during training
3. MODEL       Pick a model (decision tree, neural net, quantum circuit…)
4. TRAIN       Adjust the model's parameters to fit the training data
5. EVALUATE    Measure accuracy on the unseen test set
6. ITERATE     Tune and repeat
```

The golden rule: **always evaluate on data the model has never seen.** Otherwise you're just measuring memorisation, not learning.

## 4. Train a real classifier (scikit-learn)

Let's classify the classic **Iris** flower dataset: predict the species of a flower from 4 measurements. This is the 'hello world' of ML.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. DATA: 150 flowers, 4 features each, 3 species
iris = load_iris()
X, y = iris.data, iris.target
print("Features:", iris.feature_names)
print("Classes :", list(iris.target_names))
print("Shape   :", X.shape)

# 2. SPLIT into train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# 3 + 4. MODEL + TRAIN
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. EVALUATE on unseen data
preds = model.predict(X_test)
print(f"\nTest accuracy: {accuracy_score(y_test, preds):.2%}")
print(classification_report(y_test, preds, target_names=iris.target_names))

In a few lines, the model learned to identify flower species with high accuracy — purely from examples. Which measurements mattered most?

In [ ]:
import matplotlib.pyplot as plt

importances = model.feature_importances_
plt.barh(iris.feature_names, importances, color="teal")
plt.title("Which features the model relied on most")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

## 5. How neural networks learn

A **neural network** is a stack of simple units ('neurons'). Each connection has a **weight**. Learning = finding the weights that make the network's output match the truth.

The recipe (this is *identical in spirit* to training the quantum circuit in nb 03):

1. **Forward pass** — feed input through the network to get a prediction.
2. **Loss** — measure how wrong the prediction is (e.g. mean-squared error).
3. **Gradient** — compute how each weight affects the loss (via *backpropagation*).
4. **Update** — nudge each weight a little in the direction that lowers the loss (*gradient descent*).
5. Repeat thousands of times.

**Gradient descent** is like walking downhill in fog: feel the slope under your feet, step downhill, repeat, until you reach the bottom (minimum loss).

## 6. A tiny neural network from scratch (NumPy)

To demystify it, here is a 1-layer network learning the simple rule "output ≈ first input" — written with nothing but NumPy, so you can see every step of gradient descent.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# Toy data: 200 samples, 3 features. The 'truth' depends mostly on feature 0.
X = rng.normal(size=(200, 3))
true_w = np.array([2.0, 0.0, -1.0])
y = X @ true_w + 0.1 * rng.normal(size=200)   # target with a little noise

# Initialise weights randomly
w = rng.normal(size=3)
lr = 0.05   # learning rate (step size)

for epoch in range(200):
    # 1. Forward pass: prediction
    y_pred = X @ w
    # 2. Loss: mean squared error
    loss = np.mean((y_pred - y) ** 2)
    # 3. Gradient of the loss w.r.t. weights
    grad = (2 / len(X)) * X.T @ (y_pred - y)
    # 4. Update weights downhill
    w -= lr * grad
    if (epoch + 1) % 40 == 0:
        print(f"epoch {epoch+1:3d} | loss {loss:.4f} | weights {np.round(w, 2)}")

print("\nLearned weights:", np.round(w, 2))
print("True weights   :", true_w)

Watch the loss shrink and the learned weights converge to the true `[2, 0, -1]`. That's machine learning in its purest form — and the **exact same gradient-descent loop** trains deep networks *and* quantum circuits.

## 7. Overfitting & generalisation

The whole point of ML is to **generalise** — perform well on *new* data, not just memorise the training set.

- **Underfitting:** model too simple → poor on both train and test data.
- **Good fit:** model captures the real pattern → good on both.
- **Overfitting:** model memorises noise → great on train, poor on test.

```
Underfit          Good fit           Overfit
  ___              _/\_/\_            /\  /\  /\
 /                /      \          /  \/  \/  \   (chases every wiggle)
```

Defences against overfitting: **more data**, simpler models, *regularisation*, and always checking a held-out **validation/test set**. These same concerns apply to quantum models too.

## 8. The bridge to Quantum Machine Learning

Everything here maps directly onto Quantum ML (notebook 03):

| Classical ML | Quantum ML equivalent |
|--------------|------------------------|
| Input features `X` | Data encoded into qubits (angle/amplitude embedding) |
| Trainable weights | Rotation-gate angles in a variational circuit |
| Neural network layers | Layers of parameterised gates + entanglement |
| Loss function | Same idea (e.g. squared loss on measured outputs) |
| Gradient descent | **Same optimiser** updates the circuit's angles |
| Overfitting / generalisation | Still applies — plus new quantum-specific issues |

The big open question — *when does the quantum version actually beat the classical one?* — is the subject of notebook 06.

## Summary

- **AI ⊃ ML ⊃ Deep Learning.** ML learns rules from data instead of being hand-coded.
- Three paradigms: **supervised, unsupervised, reinforcement**.
- The workflow: **data → split → model → train → evaluate → iterate**, always testing on unseen data.
- Networks learn via **forward pass → loss → gradient → update** (gradient descent) — the same loop that trains quantum circuits.
- Beware **overfitting**; aim for **generalisation**.

**Next:** `05_Quantum_Hardware_and_Chip_Breakthroughs.ipynb` — the real machines (Google, Microsoft, IBM).

### References
- scikit-learn documentation: https://scikit-learn.org/stable/
- Google — *Machine Learning Crash Course*: https://developers.google.com/machine-learning/crash-course
- *Deep Learning* (Goodfellow, Bengio, Courville): https://www.deeplearningbook.org/